In [17]:
import pandas as pd
from tqdm import tqdm

In [18]:
results = pd.read_csv("../results/real_results/diffnaps/brca-n.tsv", sep="\t")

In [19]:
data = pd.read_csv("../../data/brca_n.csv")

In [20]:
results.columns = results.columns.str.strip()

In [21]:
target_tuple = ("Target", 1)
target_entry = data[target_tuple[0]] == target_tuple[1]
global_pattern_entry = pd.Series(False, index = data.index)

In [22]:
max_jaccard = 0
previous_entries = []
n_subgroups = 0
sizes = []
for i,r in results.iterrows():
    if r["Label"] != "tumor":
        continue
    pattern = eval(r["Pattern"])
    pattern_entry = pd.Series(True, index = data.index)
    for p in pattern:
        pattern_entry &= (data[p] == 1)
    n_subgroups += 1
    sizes.append(len(pattern))
    for e in previous_entries:
        jaccard = sum(pattern_entry & e) / sum(pattern_entry | e)
        max_jaccard = max(max_jaccard, jaccard)
    previous_entries.append(pattern_entry)
    global_pattern_entry |= pattern_entry

In [23]:
print("Model:", "DiffNAPS")
print("Dataset:", "Brca_n")
print("# Subgroups:", n_subgroups)
print("Avg. size:", sum(sizes) / n_subgroups)
print("Maximum Jaccard:", max_jaccard)

Model: DiffNAPS
Dataset: Brca_n
# Subgroups: 24
Avg. size: 6.625
Maximum Jaccard: 0.75


In [24]:
# global_pattern_entry.value_counts()
positive = (global_pattern_entry & target_entry).sum() / target_entry.sum()
negative = (global_pattern_entry & ~target_entry).sum() / (~target_entry).sum()
youden = positive- negative
youden


0.7477477477477478